# Ontario 400 Series Highway Traffic Density Map — Build Notebook

This is the real, executed notebook behind the [interactive map](https://agam2637.github.io/Ontario-400-Series-Highway-Traffic-Density-Map/) — not a cleaned-up rewrite. It includes the dead ends and bug fixes as they actually happened: a naive route filter that missed overlapping highway designations, a coordinate reference dead end (LHRS Routes had no LHRS field), a length-calculation bug from splitting lines, a forgotten highway that needed a mid-project rebuild, and — in the historical time-slider section — a `NaN`-in-JSON bug and a sneaky integer/float key mismatch that silently broke every station lookup. This is a more honest picture of the actual GIS/data engineering work than a tidy linear script would be.

**Stack:** ArcGIS Pro, Python (arcpy, pandas, numpy), Leaflet.js

**Data sources:** Ontario Road Network (ORN) Composite Service, MTO Traffic Volumes 1988-2021, MTO LHRS Base Points

**Features:** stretch-level traffic tiers (not one flat value per highway), a Highway Overview panel with per-highway stats, and a historical time slider showing 1988-2021 traffic volume evolve on a fixed color scale.

---


**Setup.** Pointing arcpy at whatever geodatabase this project's using, and telling it to overwrite outputs without complaining since I'll be re-running cells a lot.

In [11]:
import arcpy

aprx = arcpy.mp.ArcGISProject("CURRENT")
arcpy.env.workspace = aprx.defaultGeodatabase
arcpy.env.overwriteOutput = True


**First pass at pulling my 6 highways.** Hitting the Ontario Road Network Composite Service directly (it's a live feature service, no static shapefile download available) and filtering on `ROUTE_NUMBER` for an exact match. This turns out to be too naive — fixed a few cells down.

In [12]:
url = "https://services1.arcgis.com/TJH5KDher0W13Kgo/arcgis/rest/services/Ontario_Road_Network_Composite_Service_GeoHub_View_EN/FeatureServer/5"

where_clause = "ROUTE_NUMBER IN ('400', '401', '403', '404', '410', 'QEW')"

arcpy.conversion.FeatureClassToFeatureClass(
    in_features=url,
    out_path=arcpy.env.workspace,
    out_name="highways_filtered",
    where_clause=where_clause
)

<Result 'C:\\Users\\ajots\\OneDrive\\Documents\\ArcGIS\\Projects\\GTA_Highway_Traffic_Volumes\\GTA_Highway_Traffic_Volumes.gdb\\highways_filtered'>

Quick count check — 2003 segments. Reasonable, but I don't know yet that real segments are missing.

In [13]:
count = arcpy.management.GetCount("highways_filtered")
print(count)

2003


Listing every field on this layer so I know what I'm working with.

In [14]:
fields = arcpy.ListFields("highways_filtered")
for f in fields:
    print(f.name, f.type)

OBJECTID OID
Shape Geometry
ROAD_NET_ELEMENT_ID Integer
FULL_STREET_NAME String
ABBREVIATED_STREET_NAME String
ALT_STREET_NAME String
DIRECTIONAL_PREFIX String
STREET_TYPE_PREFIX String
STREET_NAME_BODY String
STREET_TYPE_SUFFIX String
DIRECTIONAL_SUFFIX String
ROAD_CLASS String
ROAD_ELEMENT_TYPE String
L_FIRST_HOUSE_NUMBER Integer
L_LAST_HOUSE_NUMBER Integer
L_HOUSE_NUMBER_STRUCTURE String
L_STANDARD_MUNICIPALITY String
R_FIRST_HOUSE_NUMBER Integer
R_LAST_HOUSE_NUMBER Integer
R_HOUSE_NUMBER_STRUCTURE String
R_STANDARD_MUNICIPALITY String
DIRECTION_OF_TRAFFIC_FLOW String
SPEED_LIMIT Integer
NUMBER_OF_LANES Integer
PAVEMENT_STATUS String
JURISDICTION String
ROUTE_NUMBER String
SHIELD_TYPE String
GEOMETRY_UPDATE_DATETIME Date
EFFECTIVE_DATETIME Date
LENGTH Double
Shape_Length Double


Noticed a gap in the map near Mississauga on the 403, so checking every distinct value actually stored in `ROUTE_NUMBER` to see if something weird is going on.

In [15]:
with arcpy.da.SearchCursor("highways_filtered", ["ROUTE_NUMBER"]) as cursor:
    unique_values = set(row[0] for row in cursor)

for v in sorted(unique_values, key=str):
    print(v)

400
401
403
404
410
QEW


Casting a wider net — searching by street name instead of route number, to see if there are more "403" segments out there than my route number filter caught.

In [16]:
url = "https://services1.arcgis.com/TJH5KDher0W13Kgo/arcgis/rest/services/Ontario_Road_Network_Composite_Service_GeoHub_View_EN/FeatureServer/5"

where_clause = "FULL_STREET_NAME LIKE '%403%' OR ALT_STREET_NAME LIKE '%403%'"

arcpy.conversion.FeatureClassToFeatureClass(
    in_features=url,
    out_path=arcpy.env.workspace,
    out_name="check_403_gap",
    where_clause=where_clause
)

count = arcpy.management.GetCount("check_403_gap")
print(count)

492


Comparing the two counts directly: exact-match `ROUTE_NUMBER = '403'` vs. anything mentioning 403 in the street name. The gap here (144 vs 492) is what confirmed something was wrong with my filter.

In [17]:
with arcpy.da.SearchCursor("highways_filtered", ["ROUTE_NUMBER"], where_clause="ROUTE_NUMBER = '403'") as cursor:
    existing_403 = sum(1 for _ in cursor)

print("Existing 403 segments:", existing_403)
print("Segments found by street name search:", count)

Existing 403 segments: 144
Segments found by street name search: 492


Printing every segment that matched on street name but doesn't have `ROUTE_NUMBER = '403'`, to see what's actually stored there instead.

In [18]:
with arcpy.da.SearchCursor("check_403_gap", ["ROUTE_NUMBER", "FULL_STREET_NAME", "ROAD_CLASS"]) as cursor:
    for row in cursor:
        if row[0] != '403':
            print(row)

('QEW; 403', 'QUEEN ELIZABETH WAY', 'Freeway')
('QEW; 403', 'QUEEN ELIZABETH WAY', 'Freeway')
('', 'HIGHWAY 403', 'Ramp')
('QEW; 403', 'QUEEN ELIZABETH WAY', 'Freeway')
('407', 'HIGHWAY 403', 'Ramp')
('403; 6', 'HIGHWAY 403', 'Freeway')
('QEW; 403', 'QUEEN ELIZABETH WAY', 'Freeway')
('QEW; 403', 'QUEEN ELIZABETH WAY', 'Freeway')
('', 'HIGHWAY 403', 'Ramp')
('403; 6', 'HIGHWAY 403', 'Freeway')
('', 'HIGHWAY 403 & HIGHWAY 407', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'HIGHWAY 403', 'Ramp')
('', 'SR403 SHORE', '')
('', 'SR403 SHORE', '')
('', 'ISLAND 4030 GEORGIAN BAY', '')
(

Tallying instead of a wall of text — how often each weird `ROUTE_NUMBER` value and `ROAD_CLASS` shows up. This revealed the real story: combined values like `'QEW; 403'` and a ton of blank-route `Ramp` segments.

In [19]:
from collections import Counter

route_vals = Counter()
road_classes = Counter()

with arcpy.da.SearchCursor("check_403_gap", ["ROUTE_NUMBER", "FULL_STREET_NAME", "ROAD_CLASS"]) as cursor:
    for row in cursor:
        if row[0] != '403':
            route_vals[row[0]] += 1
            road_classes[row[2]] += 1

print("ROUTE_NUMBER values found instead:")
for val, n in route_vals.most_common():
    print(f"  {val!r}: {n}")

print("\nROAD_CLASS values for these segments:")
for val, n in road_classes.most_common():
    print(f"  {val!r}: {n}")

ROUTE_NUMBER values found instead:
  '': 235
  'QEW; 403': 60
  '403; 6': 21
  '403; 24': 18
  '407': 5
  '6': 4
  '403; 410': 2
  '24': 2
  '401': 1

ROAD_CLASS values for these segments:
  'Ramp': 231
  'Freeway': 101
  'Resource / Recreation': 9
  '': 3
  'Arterial': 2
  'Local / Strata': 1
  'Expressway / Highway': 1


**The actual fix.** Using `LIKE` to catch combined route values, restricted to `ROAD_CLASS = 'Freeway'` so I don't pull in unrelated roads that just happen to contain the same digits.

In [20]:
url = "https://services1.arcgis.com/TJH5KDher0W13Kgo/arcgis/rest/services/Ontario_Road_Network_Composite_Service_GeoHub_View_EN/FeatureServer/5"

target_routes = {"400", "401", "403", "404", "410", "QEW"}

like_clause = " OR ".join([f"ROUTE_NUMBER LIKE '%{r}%'" for r in target_routes])
where_clause = f"({like_clause}) AND ROAD_CLASS = 'Freeway'"

arcpy.conversion.FeatureClassToFeatureClass(
    in_features=url,
    out_path=arcpy.env.workspace,
    out_name="highways_mainline",
    where_clause=where_clause
)

print(arcpy.management.GetCount("highways_mainline"))

2090


Ramps mostly don't carry a route number at all, so matching them separately by street name instead, keeping only `ROAD_CLASS = 'Ramp'`.

In [21]:
street_names = ["HIGHWAY 400", "HIGHWAY 401", "HIGHWAY 403", "HIGHWAY 404", "HIGHWAY 410", "QUEEN ELIZABETH WAY"]

street_clause = " OR ".join([f"FULL_STREET_NAME LIKE '%{s}%'" for s in street_names])
where_clause = f"({street_clause}) AND ROAD_CLASS = 'Ramp'"

arcpy.conversion.FeatureClassToFeatureClass(
    in_features=url,
    out_path=arcpy.env.workspace,
    out_name="highways_ramps",
    where_clause=where_clause
)

print(arcpy.management.GetCount("highways_ramps"))

2421


Tagging both layers with a `segment_type` field (Mainline vs Ramp) and merging into one layer — my fixed "v2" highway layer.

In [22]:
arcpy.management.AddField("highways_mainline", "segment_type", "TEXT", field_length=20)
arcpy.management.CalculateField("highways_mainline", "segment_type", "'Mainline'")

arcpy.management.AddField("highways_ramps", "segment_type", "TEXT", field_length=20)
arcpy.management.CalculateField("highways_ramps", "segment_type", "'Ramp'")

arcpy.management.Merge(
    inputs=["highways_mainline", "highways_ramps"],
    output="highways_filtered_v2"
)

print(arcpy.management.GetCount("highways_filtered_v2"))

4511


Sanity check on the merge — nothing unrelated snuck in through the wider `LIKE` filter.

In [23]:
from collections import Counter

with arcpy.da.SearchCursor("highways_filtered_v2", ["ROUTE_NUMBER", "segment_type"]) as cursor:
    tally = Counter((row[0], row[1]) for row in cursor)

for (route, seg_type), n in sorted(tally.items(), key=lambda x: -x[1])[:20]:
    print(route, seg_type, n)

﻿ Ramp 2301
401 Mainline 1180
400 Mainline 254
QEW Mainline 229
403 Mainline 134
404 Mainline 93
QEW; 403 Mainline 60
410 Mainline 55
401 Ramp 40
400; 69 Mainline 35
403; 6 Mainline 21
403; 24 Mainline 18
6 Ramp 13
403 Ramp 10
137 Ramp 7
400 Ramp 6
420 Ramp 6
407 Ramp 5
35; 115 Ramp 5
406 Ramp 5


**Starting on the traffic volume side.** First look at the MTO AADT CSV — columns, shape, structure.

In [24]:
import pandas as pd

csv_path = r"data raw\Traffic_Volumes_1988-2021.csv"
aadt_df = pd.read_csv(csv_path)

print(aadt_df.shape)
print(aadt_df.columns.tolist())
aadt_df.head()

<class 'FileNotFoundError'>: [Errno 2] No such file or directory: 'data raw\\Traffic_Volumes_1988-2021.csv'

Trying to figure out MTO's internal `Hwy No` codes by searching location text for my target highway names. This approach is flawed (it also catches highways just *mentioned* nearby) — kept for the record, real fix is next cell.

In [ ]:
# Look for rows whose location description mentions our target highways by name,
# and see what Hwy No value those rows actually have
keywords = ["QEW", "401", "400", "403", "404", "410"]

for kw in keywords:
    matches = aadt_df[aadt_df["Location Description"].str.contains(kw, na=False)]
    print(f"--- '{kw}' ---")
    print(matches[["Hwy No", "Hwy No Suffix", "Hwy. Type", "Location Description"]].drop_duplicates().head(10))
    print()

**The real check.** Filtering directly by candidate `Hwy No` and reading the location descriptions to see if they trace a believable route. This is how I found the QEW is secretly `Hwy No = 1`.

In [ ]:
# Check each candidate Hwy No directly and see if the locations trace a believable route
for hwy in [1, 400, 401, 403, 404, 410]:
    subset = aadt_df[aadt_df["Hwy No"] == hwy]
    print(f"--- Hwy No = {hwy} ({len(subset)} rows) ---")
    print(subset["Location Description"].drop_duplicates().head(8).tolist())
    print()

Filtering AADT data down to my 6 highways using the confirmed real codes.

In [ ]:
target_hwy_codes = [1, 400, 401, 403, 404, 410]  # 1 = QEW

aadt_filtered = aadt_df[aadt_df["Hwy No"].isin(target_hwy_codes)].copy()

print(aadt_filtered.shape)
print(aadt_filtered["Hwy No"].value_counts())

Checking available years — found no 2020 (COVID gap, probably) and a bogus `9999` placeholder.

In [ ]:
print(sorted(aadt_filtered["Year"].unique()))

Picking 2021 as the most recent real year.

In [ ]:
aadt_2021 = aadt_filtered[aadt_filtered["Year"] == 2021].copy()
print(aadt_2021.shape)
aadt_2021[["Hwy No", "Location Description", "AADT", "LHRS", "Offset"]].head(10)

Checking the AADT column's dtype before doing math on it — turned out to be comma-formatted text, not a clean number.

In [ ]:
print(aadt_2021["AADT"].dtype)
print(aadt_2021["AADT"].head(10))

First attempt at cleaning AADT into a float.

In [ ]:
aadt_2021["AADT_clean"] = aadt_2021["AADT"].astype(str).str.replace(",", "").astype(float)

Redid the cleaning with `9999` explicitly excluded too, plus `.describe()` to sanity check the distribution.

In [ ]:
aadt_2021 = aadt_2021[aadt_2021["Year"] != 9999].copy()  # just in case, belt-and-suspenders

aadt_2021["AADT_clean"] = aadt_2021["AADT"].astype(str).str.replace(",", "").astype(float)

print(aadt_2021["AADT_clean"].describe())

**Dead end, but useful.** Downloaded MTO's LHRS Routes shapefile hoping to place AADT stations precisely — turned out to have zero LHRS fields, just one line per highway. Not what I needed.

In [ ]:
lhrs_routes_path = r"data raw\lhrs_routes_july2015\LHRS_Routes_July2015.shp"

arcpy.management.CopyFeatures(lhrs_routes_path, "lhrs_routes")

fields = arcpy.ListFields("lhrs_routes")
for f in fields:
    print(f.name, f.type)

Checking what highway codes this Routes file uses, before giving up on it.

In [ ]:
with arcpy.da.SearchCursor("lhrs_routes", ["HWY_2015"]) as cursor:
    unique_hwy = sorted(set(row[0] for row in cursor), key=str)

print(unique_hwy[:30])
print(len(unique_hwy))

Confirmed my 6 highways are present here, but this file still doesn't have what I actually need.

In [ ]:
target = ['1', '400', '401', '403', '404', '410']
found = [h for h in unique_hwy if h in target]
print(found)

**The file that actually works.** MTO's LHRS Base Points shapefile — has `LHRS`, `OFFSET`, and direct `LATITUDE`/`LONGITUDE`.

In [ ]:
base_points_path = r"data raw\LHRS_Base_Points_July2015.shp"

arcpy.management.CopyFeatures(base_points_path, "lhrs_base_points")

fields = arcpy.ListFields("lhrs_base_points")
for f in fields:
    print(f.name, f.type)

Checking how many of my 327 AADT stations match a `(LHRS, Offset)` pair in Base Points. First run — redone after a notebook reset a few cells down.

In [ ]:
with arcpy.da.SearchCursor("lhrs_base_points", ["LHRS", "OFFSET", "HWY"]) as cursor:
    base_keys = set((row[0], row[1]) for row in cursor)

# Build the same key from your AADT 2021 data
aadt_keys = set(zip(aadt_2021["LHRS"], aadt_2021["Offset"]))

matched = aadt_keys & base_keys
print(f"AADT stations: {len(aadt_keys)}")
print(f"Base points: {len(base_keys)}")
print(f"Matched: {len(matched)}")

Notebook kernel reset (closed ArcGIS Pro, went idle, who knows) and wiped my variables — reloading from scratch to get back to where I was.

In [ ]:
import pandas as pd

csv_path = r"data raw\Traffic_Volumes_1988-2021.csv"
aadt_df = pd.read_csv(csv_path)

target_hwy_codes = [1, 400, 401, 403, 404, 410]  # 1 = QEW
aadt_filtered = aadt_df[aadt_df["Hwy No"].isin(target_hwy_codes)].copy()

aadt_2021 = aadt_filtered[aadt_filtered["Year"] == 2021].copy()
aadt_2021["AADT_clean"] = aadt_2021["AADT"].astype(str).str.replace(",", "").astype(float)

print(aadt_2021.shape)

This reload broke with `FileNotFoundError` — checking what directory Python actually thinks it's in.

In [ ]:
import os
print(os.getcwd())

**Fixing the path for real.** Building an absolute path off `aprx.homeFolder` instead of a fragile relative one.

In [ ]:
import arcpy
import os

aprx = arcpy.mp.ArcGISProject("CURRENT")
project_folder = aprx.homeFolder
print(project_folder)

csv_path = os.path.join(project_folder, "data raw", "Traffic_Volumes_1988-2021.csv")
print(csv_path)
print(os.path.exists(csv_path))  # should print True

Reloading the AADT CSV with the new reliable path.

In [ ]:
import pandas as pd

aadt_df = pd.read_csv(csv_path)

target_hwy_codes = [1, 400, 401, 403, 404, 410]  # 1 = QEW
aadt_filtered = aadt_df[aadt_df["Hwy No"].isin(target_hwy_codes)].copy()

aadt_2021 = aadt_filtered[aadt_filtered["Year"] == 2021].copy()
aadt_2021["AADT_clean"] = aadt_2021["AADT"].astype(str).str.replace(",", "").astype(float)

print(aadt_2021.shape)

Re-running the join match check — 326 out of 327 matched.

In [ ]:
with arcpy.da.SearchCursor("lhrs_base_points", ["LHRS", "OFFSET", "HWY"]) as cursor:
    base_keys = set((row[0], row[1]) for row in cursor)

aadt_keys = set(zip(aadt_2021["LHRS"], aadt_2021["Offset"]))

matched = aadt_keys & base_keys
print(f"AADT stations: {len(aadt_keys)}")
print(f"Base points: {len(base_keys)}")
print(f"Matched: {len(matched)}")

Curious about the one unmatched station — a single 401 station near Wonderland Rd, probably added after this 2015 Base Points snapshot.

In [ ]:
unmatched = aadt_keys - base_keys
print(unmatched)

# Look up that station's details in your AADT data
lhrs_val, offset_val = list(unmatched)[0]
row = aadt_2021[(aadt_2021["LHRS"] == lhrs_val) & (aadt_2021["Offset"] == offset_val)]
print(row[["Hwy No", "Location Description", "AADT_clean", "LHRS", "Offset"]])

Same as above, re-ran it.

In [25]:
unmatched = aadt_keys - base_keys
print(unmatched)

# Look up that station's details in your AADT data
lhrs_val, offset_val = list(unmatched)[0]
row = aadt_2021[(aadt_2021["LHRS"] == lhrs_val) & (aadt_2021["Offset"] == offset_val)]
print(row[["Hwy No", "Location Description", "AADT_clean", "LHRS", "Offset"]])

{(47828, 0.0)}
       Hwy No  Location Description  AADT_clean   LHRS  Offset
41013     401  WONDERLAND RD IC-180     29300.0  47828     0.0


Building a `(LHRS, Offset) -> AADT` lookup and writing it onto the Base Points layer as a new field.

In [26]:
# Build a lookup dict: (LHRS, Offset) -> AADT_clean
aadt_lookup = {
    (row["LHRS"], row["Offset"]): row["AADT_clean"]
    for _, row in aadt_2021.iterrows()
}

# Add a new field to lhrs_base_points and populate it
arcpy.management.AddField("lhrs_base_points", "AADT_2021", "DOUBLE")

with arcpy.da.UpdateCursor("lhrs_base_points", ["LHRS", "OFFSET", "AADT_2021"]) as cursor:
    for row in cursor:
        key = (row[0], row[1])
        if key in aadt_lookup:
            row[2] = aadt_lookup[key]
            cursor.updateRow(row)

Filtering Base Points to my 6 highways, keeping only ones with an actual AADT value — this becomes `aadt_stations_2021`.

In [27]:
where_clause = "HWY IN ('1','400','401','403','404','410') AND AADT_2021 IS NOT NULL"

arcpy.management.SelectLayerByAttribute("lhrs_base_points", "NEW_SELECTION", where_clause)
arcpy.management.CopyFeatures("lhrs_base_points", "aadt_stations_2021")

count = arcpy.management.GetCount("aadt_stations_2021")
print(count)

319


Spot-checking coordinates and AADT values before building anything else on top.

In [28]:
with arcpy.da.SearchCursor("aadt_stations_2021", ["HWY", "LATITUDE", "LONGITUDE", "AADT_2021"]) as cursor:
    for i, row in enumerate(cursor):
        if i < 5:
            print(row)

('401', 43.229106, -80.60205599, 50500.0)
('400', 43.84668588, -79.54847475, 140200.0)
('401', 42.48247219, -81.91116389, 22800.0)
('401', 42.24252476, -82.55214227, 22500.0)
('401', 44.917106, -75.197659, 18500.0)


Pulling out just the Mainline segments — these are what actually get split into stretches.

In [29]:
where_clause = "segment_type = 'Mainline'"
arcpy.management.SelectLayerByAttribute("highways_filtered_v2", "NEW_SELECTION", where_clause)
arcpy.management.CopyFeatures("highways_filtered_v2", "highways_mainline_only")

<Result 'C:\\Users\\ajots\\OneDrive\\Documents\\ArcGIS\\Projects\\GTA_Highway_Traffic_Volumes\\GTA_Highway_Traffic_Volumes.gdb\\highways_mainline_only'>

**Splitting the highway at each station point** so different parts of the same highway can get different colors, instead of one flat value for the whole route.

In [30]:
arcpy.management.SplitLineAtPoint(
    in_features="highways_mainline_only",
    point_features="aadt_stations_2021",
    out_feature_class="highways_split",
    search_radius="100 Meters"
)

count = arcpy.management.GetCount("highways_split")
print(count)

2816


First attempt joining each stretch to its nearest station, 500m radius. Left 878 unmatched — rural stretches can be way more than 500m from the nearest station.

In [31]:
arcpy.analysis.SpatialJoin(
    target_features="highways_split",
    join_features="aadt_stations_2021",
    out_feature_class="highways_with_aadt",
    join_operation="JOIN_ONE_TO_ONE",
    match_option="CLOSEST",
    search_radius="500 Meters"
)

count = arcpy.management.GetCount("highways_with_aadt")
print(count)

# Quick check: how many got a real AADT value vs came back empty
with arcpy.da.SearchCursor("highways_with_aadt", ["AADT_2021"]) as cursor:
    null_count = sum(1 for row in cursor if row[0] is None)
print(f"Segments with no matched AADT: {null_count}")

2816
Segments with no matched AADT: 878


**Fixed** by widening to 50km. Every stretch matched.

In [32]:
arcpy.analysis.SpatialJoin(
    target_features="highways_split",
    join_features="aadt_stations_2021",
    out_feature_class="highways_with_aadt_v2",
    join_operation="JOIN_ONE_TO_ONE",
    match_option="CLOSEST",
    search_radius="50000 Meters"  # 50 km - should comfortably catch even the sparsest rural gaps
)

count = arcpy.management.GetCount("highways_with_aadt_v2")
print(count)

with arcpy.da.SearchCursor("highways_with_aadt_v2", ["AADT_2021"]) as cursor:
    null_count = sum(1 for row in cursor if row[0] is None)
print(f"Segments with no matched AADT: {null_count}")

2816
Segments with no matched AADT: 0


Sanity-checking the AADT distribution on the split data — matched the original station-level stats.

In [33]:
# Add highways_with_aadt_v2 to your map and visually confirm color/value makes sense once symbolized later.
# For now, just check the AADT distribution looks reasonable across the new stretch-level data:

with arcpy.da.SearchCursor("highways_with_aadt_v2", ["AADT_2021"]) as cursor:
    values = [row[0] for row in cursor]

import numpy as np
values = np.array(values)
print(f"Count: {len(values)}")
print(f"Min: {values.min()}, Max: {values.max()}")
print(f"25th pct: {np.percentile(values, 25):.0f}")
print(f"Median: {np.percentile(values, 50):.0f}")
print(f"75th pct: {np.percentile(values, 75):.0f}")

Count: 2816
Min: 8100.0, Max: 480900.0
25th pct: 29300
Median: 80400
75th pct: 176200


Calculating tercile break points for Low/Medium/High.

In [34]:
import numpy as np

low_threshold = np.percentile(values, 33.33)
high_threshold = np.percentile(values, 66.67)

print(f"Low/Medium boundary: {low_threshold:.0f}")
print(f"Medium/High boundary: {high_threshold:.0f}")

Low/Medium boundary: 40100
Medium/High boundary: 130100


Writing the `traffic_tier` label onto every stretch.

In [35]:
arcpy.management.AddField("highways_with_aadt_v2", "traffic_tier", "TEXT", field_length=20)

with arcpy.da.UpdateCursor("highways_with_aadt_v2", ["AADT_2021", "traffic_tier"]) as cursor:
    for row in cursor:
        aadt = row[0]
        if aadt is None:
            row[1] = "No Data"
        elif aadt <= low_threshold:
            row[1] = "Low"
        elif aadt <= high_threshold:
            row[1] = "Medium"
        else:
            row[1] = "High"
        cursor.updateRow(row)

Tallying tier counts — came out almost perfectly even.

In [36]:
from collections import Counter

with arcpy.da.SearchCursor("highways_with_aadt_v2", ["traffic_tier"]) as cursor:
    tally = Counter(row[0] for row in cursor)

for tier, n in tally.items():
    print(tier, n)

Medium 937
High 938
Low 941


Pulling Ramp segments back in with a fixed `'Ramp'` tier, merging into `highways_classified`.

In [37]:
where_clause = "segment_type = 'Ramp'"
arcpy.management.SelectLayerByAttribute("highways_filtered_v2", "NEW_SELECTION", where_clause)
arcpy.management.CopyFeatures("highways_filtered_v2", "highways_ramps_only")

arcpy.management.AddField("highways_ramps_only", "traffic_tier", "TEXT", field_length=20)
arcpy.management.CalculateField("highways_ramps_only", "traffic_tier", "'Ramp'")

arcpy.management.Merge(
    inputs=["highways_with_aadt_v2", "highways_ramps_only"],
    output="highways_classified"
)

count = arcpy.management.GetCount("highways_classified")
print(count)

5237


Symbology in ArcGIS Pro wasn't showing `traffic_tier`, so double-checking the field actually exists.

In [38]:
fields = arcpy.ListFields("highways_classified")
for f in fields:
    print(f.name, f.type)

OBJECTID OID
Shape Geometry
Join_Count Integer
TARGET_FID Integer
ROAD_NET_ELEMENT_ID Integer
FULL_STREET_NAME String
ABBREVIATED_STREET_NAME String
ALT_STREET_NAME String
DIRECTIONAL_PREFIX String
STREET_TYPE_PREFIX String
STREET_NAME_BODY String
STREET_TYPE_SUFFIX String
DIRECTIONAL_SUFFIX String
ROAD_CLASS String
ROAD_ELEMENT_TYPE String
L_FIRST_HOUSE_NUMBER Integer
L_LAST_HOUSE_NUMBER Integer
L_HOUSE_NUMBER_STRUCTURE String
L_STANDARD_MUNICIPALITY String
R_FIRST_HOUSE_NUMBER Integer
R_LAST_HOUSE_NUMBER Integer
R_HOUSE_NUMBER_STRUCTURE String
R_STANDARD_MUNICIPALITY String
DIRECTION_OF_TRAFFIC_FLOW String
SPEED_LIMIT Integer
NUMBER_OF_LANES Integer
PAVEMENT_STATUS String
JURISDICTION String
ROUTE_NUMBER String
SHIELD_TYPE String
GEOMETRY_UPDATE_DATETIME Date
EFFECTIVE_DATETIME Date
LENGTH Double
segment_type String
ORIG_FID Integer
ORIG_SEQ Integer
MTO_REGION String
MTO_DISTRI String
OPP_DISTRI String
COUNTY Integer
HWY String
LHRS Integer
LENGTH_1 Double
OFFSET Integer
DESCRIPTIO S

First attempt at clearing a stray selection (leftover cyan highlight from all my `SelectLayerByAttribute` calls). Crashed — `SELECTIONSET` isn't a valid property in this arcpy version.

In [39]:
aprx = arcpy.mp.ArcGISProject("CURRENT")
m = aprx.activeMap

for lyr in m.listLayers():
    if lyr.supports("SELECTIONSET"):
        arcpy.management.SelectLayerByAttribute(lyr, "CLEAR_SELECTION")

<class 'ValueError'>: Invalid value for layer_property: 'SELECTIONSET' (choices are: ['BRIGHTNESS', 'CONNECTIONPROPERTIES', 'CONTRAST', 'DATASOURCE', 'DEFINITIONQUERY', 'IS3DLAYER', 'ISBASEMAPLAYER', 'ISBROKEN', 'ISFEATURELAYER', 'ISGRAPHICSLAYER', 'ISGROUPLAYER', 'ISNETWORKANALYSTLAYER', 'ISNETWORKDATASETLAYER', 'ISPARCELFABRICLAYER', 'ISRASTERLAYER', 'ISSCENELAYER', 'ISTIMEENABLED', 'ISWEBLAYER', 'LONGNAME', 'MAXTHRESHOLD', 'METADATA', 'MINTHRESHOLD', 'NAME', 'SHOWLABELS', 'SYMBOLOGY', 'TRANSPARENCY', 'TIME', 'URI', 'VISIBLE'])

**Fixed** — just try clearing selection on every layer and skip whichever ones don't support it.

In [40]:
aprx = arcpy.mp.ArcGISProject("CURRENT")
m = aprx.activeMap

for lyr in m.listLayers():
    if lyr.isFeatureLayer:
        try:
            arcpy.management.SelectLayerByAttribute(lyr, "CLEAR_SELECTION")
        except Exception as e:
            print(f"Skipped {lyr.name}: {e}")

Realized I'd forgotten Highway 427 entirely. Checking how it's coded — turned out simple, just `427` directly.

In [41]:
matches = aadt_df[aadt_df["Location Description"].str.contains("427", na=False)]
print(matches[["Hwy No", "Location Description"]].drop_duplicates().head(10))

# Also check directly
subset = aadt_df[aadt_df["Hwy No"] == 427]
print(subset["Location Description"].drop_duplicates().head(8).tolist())

       Hwy No                  Location Description
1823        1  HWY 427 IC-139 ETOBICOKE START OF NA
22581      27                     HWY 427/401 RAMPS
39767     401                        HWY 427 IC-352
['COULES CT ETOBICOKE START OF NA', 'EVANS AV IC ETOBICOKE', 'QEW IC START OF COMPLEX FREEWAY', 'HWY 5 IC DUNDAS ST ETOBICOKE', 'BURNHAMTHORPE RD IC ETOBICOKE', 'RATHBURN RD IC ETOBICOKE', 'HWY 401 IC', 'DIXON RD IC ETOBICOKE']


Rebuilding the highway pull with 427 added.

In [42]:
url = "https://services1.arcgis.com/TJH5KDher0W13Kgo/arcgis/rest/services/Ontario_Road_Network_Composite_Service_GeoHub_View_EN/FeatureServer/5"

target_routes = {"400", "401", "403", "404", "410", "427", "QEW"}

like_clause = " OR ".join([f"ROUTE_NUMBER LIKE '%{r}%'" for r in target_routes])
where_clause = f"({like_clause}) AND ROAD_CLASS = 'Freeway'"
arcpy.conversion.FeatureClassToFeatureClass(url, arcpy.env.workspace, "highways_mainline", where_clause)

street_names = ["HIGHWAY 400", "HIGHWAY 401", "HIGHWAY 403", "HIGHWAY 404", "HIGHWAY 410", "HIGHWAY 427", "QUEEN ELIZABETH WAY"]
street_clause = " OR ".join([f"FULL_STREET_NAME LIKE '%{s}%'" for s in street_names])
where_clause = f"({street_clause}) AND ROAD_CLASS = 'Ramp'"
arcpy.conversion.FeatureClassToFeatureClass(url, arcpy.env.workspace, "highways_ramps", where_clause)

arcpy.management.AddField("highways_mainline", "segment_type", "TEXT", field_length=20)
arcpy.management.CalculateField("highways_mainline", "segment_type", "'Mainline'")
arcpy.management.AddField("highways_ramps", "segment_type", "TEXT", field_length=20)
arcpy.management.CalculateField("highways_ramps", "segment_type", "'Ramp'")

arcpy.management.Merge(["highways_mainline", "highways_ramps"], "highways_filtered_v2")
print(arcpy.management.GetCount("highways_filtered_v2"))

4785


Rebuilding the AADT filtering with 427's code included.

In [43]:
target_hwy_codes = [1, 400, 401, 403, 404, 410, 427]  # 1 = QEW
aadt_filtered = aadt_df[aadt_df["Hwy No"].isin(target_hwy_codes)].copy()
aadt_2021 = aadt_filtered[aadt_filtered["Year"] == 2021].copy()
aadt_2021["AADT_clean"] = aadt_2021["AADT"].astype(str).str.replace(",", "").astype(float)
print(aadt_2021.shape)

(338, 30)


Re-running the cell above — same rebuild, ran it twice.

In [44]:
url = "https://services1.arcgis.com/TJH5KDher0W13Kgo/arcgis/rest/services/Ontario_Road_Network_Composite_Service_GeoHub_View_EN/FeatureServer/5"

target_routes = {"400", "401", "403", "404", "410", "427", "QEW"}

like_clause = " OR ".join([f"ROUTE_NUMBER LIKE '%{r}%'" for r in target_routes])
where_clause = f"({like_clause}) AND ROAD_CLASS = 'Freeway'"
arcpy.conversion.FeatureClassToFeatureClass(url, arcpy.env.workspace, "highways_mainline", where_clause)

street_names = ["HIGHWAY 400", "HIGHWAY 401", "HIGHWAY 403", "HIGHWAY 404", "HIGHWAY 410", "HIGHWAY 427", "QUEEN ELIZABETH WAY"]
street_clause = " OR ".join([f"FULL_STREET_NAME LIKE '%{s}%'" for s in street_names])
where_clause = f"({street_clause}) AND ROAD_CLASS = 'Ramp'"
arcpy.conversion.FeatureClassToFeatureClass(url, arcpy.env.workspace, "highways_ramps", where_clause)

arcpy.management.AddField("highways_mainline", "segment_type", "TEXT", field_length=20)
arcpy.management.CalculateField("highways_mainline", "segment_type", "'Mainline'")
arcpy.management.AddField("highways_ramps", "segment_type", "TEXT", field_length=20)
arcpy.management.CalculateField("highways_ramps", "segment_type", "'Ramp'")

arcpy.management.Merge(["highways_mainline", "highways_ramps"], "highways_filtered_v2")
print(arcpy.management.GetCount("highways_filtered_v2"))

4785


Same rebuild again.

In [45]:
target_hwy_codes = [1, 400, 401, 403, 404, 410, 427]  # 1 = QEW
aadt_filtered = aadt_df[aadt_df["Hwy No"].isin(target_hwy_codes)].copy()
aadt_2021 = aadt_filtered[aadt_filtered["Year"] == 2021].copy()
aadt_2021["AADT_clean"] = aadt_2021["AADT"].astype(str).str.replace(",", "").astype(float)
print(aadt_2021.shape)

aadt_lookup = {(row["LHRS"], row["Offset"]): row["AADT_clean"] for _, row in aadt_2021.iterrows()}

# Re-add the field fresh (in case it still has old values from before)
arcpy.management.AddField("lhrs_base_points", "AADT_2021_v2", "DOUBLE")
with arcpy.da.UpdateCursor("lhrs_base_points", ["LHRS", "OFFSET", "AADT_2021_v2"]) as cursor:
    for row in cursor:
        key = (row[0], row[1])
        if key in aadt_lookup:
            row[2] = aadt_lookup[key]
            cursor.updateRow(row)

where_clause = "HWY IN ('1','400','401','403','404','410','427') AND AADT_2021_v2 IS NOT NULL"
arcpy.management.SelectLayerByAttribute("lhrs_base_points", "NEW_SELECTION", where_clause)
arcpy.management.CopyFeatures("lhrs_base_points", "aadt_stations_2021_v2")
print(arcpy.management.GetCount("aadt_stations_2021_v2"))

(338, 30)
330


Rebuilding the split-at-stations step with the 427-inclusive data.

In [46]:
arcpy.management.SelectLayerByAttribute("highways_filtered_v2", "NEW_SELECTION", "segment_type = 'Mainline'")
arcpy.management.CopyFeatures("highways_filtered_v2", "highways_mainline_only")

arcpy.management.SplitLineAtPoint(
    in_features="highways_mainline_only",
    point_features="aadt_stations_2021_v2",
    out_feature_class="highways_split",
    search_radius="100 Meters"
)
print(arcpy.management.GetCount("highways_split"))

arcpy.analysis.SpatialJoin(
    target_features="highways_split",
    join_features="aadt_stations_2021_v2",
    out_feature_class="highways_with_aadt_v2",
    join_operation="JOIN_ONE_TO_ONE",
    match_option="CLOSEST",
    search_radius="50000 Meters"
)
print(arcpy.management.GetCount("highways_with_aadt_v2"))

with arcpy.da.SearchCursor("highways_with_aadt_v2", ["AADT_2021_v2"]) as cursor:
    null_count = sum(1 for row in cursor if row[0] is None)
print(f"Segments with no matched AADT: {null_count}")

2945
2945
Segments with no matched AADT: 0


Recomputing tercile breakpoints now that 427's traffic is in the pool (shifted slightly, as expected), and reclassifying.

In [47]:
with arcpy.da.SearchCursor("highways_with_aadt_v2", ["AADT_2021_v2"]) as cursor:
    values = [row[0] for row in cursor]

import numpy as np
values = np.array(values)

low_threshold = np.percentile(values, 33.33)
high_threshold = np.percentile(values, 66.67)
print(f"Low/Medium boundary: {low_threshold:.0f}")
print(f"Medium/High boundary: {high_threshold:.0f}")

arcpy.management.AddField("highways_with_aadt_v2", "traffic_tier", "TEXT", field_length=20)
with arcpy.da.UpdateCursor("highways_with_aadt_v2", ["AADT_2021_v2", "traffic_tier"]) as cursor:
    for row in cursor:
        aadt = row[0]
        if aadt is None:
            row[1] = "No Data"
        elif aadt <= low_threshold:
            row[1] = "Low"
        elif aadt <= high_threshold:
            row[1] = "Medium"
        else:
            row[1] = "High"
        cursor.updateRow(row)

Low/Medium boundary: 41200
Medium/High boundary: 143200


Merging the 427-inclusive ramps back in to build the final `highways_classified_v2`.

In [48]:
arcpy.management.SelectLayerByAttribute("highways_filtered_v2", "NEW_SELECTION", "segment_type = 'Ramp'")
arcpy.management.CopyFeatures("highways_filtered_v2", "highways_ramps_only")
arcpy.management.AddField("highways_ramps_only", "traffic_tier", "TEXT", field_length=20)
arcpy.management.CalculateField("highways_ramps_only", "traffic_tier", "'Ramp'")

arcpy.management.Merge(["highways_with_aadt_v2", "highways_ramps_only"], "highways_classified_v2")
print(arcpy.management.GetCount("highways_classified_v2"))

5539


Clearing selection again before symbolizing.

In [49]:
aprx = arcpy.mp.ArcGISProject("CURRENT")
m = aprx.activeMap
for lyr in m.listLayers():
    if lyr.isFeatureLayer:
        try:
            arcpy.management.SelectLayerByAttribute(lyr, "CLEAR_SELECTION")
        except Exception as e:
            pass

Listing every layer in the map's Contents pane — about 14 scratch layers had piled up by this point.

In [50]:
import arcpy

aprx = arcpy.mp.ArcGISProject("CURRENT")
m = aprx.activeMap

# Layer names to keep visible - everything else gets removed from the map
keep_layers = ["highways_classified_v2"]

# List what's currently in the map first, so you can see what will be removed
for lyr in m.listLayers():
    print(lyr.name)

aadt_stations_2021_v2
aadt_stations_2021
lhrs_base_points
highways_classified_v2
highways_ramps_only
highways_with_aadt_v2
highways_split
highways_mainline_only
highways_filtered_v2
highways_ramps
highways_mainline
highways_classified
highways_with_aadt
check_403_gap
highways_filtered
lhrs_routes
World Topographic Map
World Hillshade


**Cleaning up** — removing everything except the final classified layer and basemaps. Only touches the map view, not the geodatabase.

In [51]:
keep_layers = ["highways_classified_v2", "World Topographic Map", "World Hillshade"]

for lyr in m.listLayers():
    if lyr.name not in keep_layers:
        m.removeLayer(lyr)

# Confirm what's left
for lyr in m.listLayers():
    print(lyr.name)
    

highways_classified_v2
World Topographic Map
World Hillshade


Checking the coordinate system before exporting to a web map.

In [52]:
desc = arcpy.Describe("highways_classified_v2")
print(desc.spatialReference.name)

GCS_North_American_1983


Reprojecting to WGS84 so the data lines up correctly in Leaflet.

In [53]:
arcpy.management.Project(
    in_dataset="highways_classified_v2",
    out_dataset="highways_classified_wgs84",
    out_coor_system=arcpy.SpatialReference(4326)  # WGS84
)

<Result 'C:\\Users\\ajots\\OneDrive\\Documents\\ArcGIS\\Projects\\GTA_Highway_Traffic_Volumes\\GTA_Highway_Traffic_Volumes.gdb\\highways_classified_wgs84'>

Exporting to GeoJSON — the file the web map actually reads. (First export landed in `data raw` by mistake, moved to `data processed` for the real workflow.)

In [54]:
arcpy.conversion.FeaturesToJSON(
    in_features="highways_classified_wgs84",
    out_json_file=r"data raw\highways_classified.geojson",
    geoJSON="GEOJSON"
)

<Result 'data raw\\highways_classified.geojson'>

First attempt merging the GeoJSON into the HTML template as embedded data, so the map works via double-click with no local server.

In [55]:
import json

# Read your GeoJSON
with open(r"data processed\highways_classified.geojson", "r", encoding="utf-8") as f:
    geojson_data = json.load(f)

# Read the HTML template
with open(r"data processed\highways_classified.html", "r", encoding="utf-8") as f:
    html_content = f.read()

# Build the embedded data script tag
embed_script = f"<script>window.EMBEDDED_HIGHWAY_DATA = {json.dumps(geojson_data)};</script>\n"

# Insert it right before the closing </body> tag
merged_html = html_content.replace("</body>", embed_script + "</body>")

# Save as a new standalone file
with open(r"data processed\highways_classified_standalone.html", "w", encoding="utf-8") as f:
    f.write(merged_html)

print("Done - standalone file created")

Done - standalone file created


Re-running the same merge attempt.

In [56]:
import json

with open(r"data processed\highways_classified.geojson", "r", encoding="utf-8") as f:
    geojson_data = json.load(f)

with open(r"data processed\highways_classified.html", "r", encoding="utf-8") as f:
    html_content = f.read()

embed_script = f"<script>window.EMBEDDED_HIGHWAY_DATA = {json.dumps(geojson_data)};</script>\n"
merged_html = html_content.replace("</body>", embed_script + "</body>")

with open(r"data processed\highways_classified_standalone.html", "w", encoding="utf-8") as f:
    f.write(merged_html)

print("Done - standalone file created")

<class 'FileNotFoundError'>: [Errno 2] No such file or directory: 'data processed\\highways_classified.html'

Switching to the reliable absolute-path pattern since a plain relative path couldn't find the HTML template.

In [57]:
import json
import arcpy
import os

aprx = arcpy.mp.ArcGISProject("CURRENT")
project_folder = aprx.homeFolder

geojson_path = os.path.join(project_folder, "data processed", "highways_classified.geojson")
html_path = os.path.join(project_folder, "data processed", "highways_classified.html")
output_path = os.path.join(project_folder, "data processed", "highways_classified_standalone.html")

print(os.path.exists(geojson_path))
print(os.path.exists(html_path))

True
False


Re-running the path setup — this is where I found the HTML template genuinely hadn't been moved into the right folder yet.

In [58]:
import json
import arcpy
import os

aprx = arcpy.mp.ArcGISProject("CURRENT")
project_folder = aprx.homeFolder

geojson_path = os.path.join(project_folder, "data processed", "highways_classified.geojson")
html_path = os.path.join(project_folder, "data processed", "highways_classified.html")
output_path = os.path.join(project_folder, "data processed", "highways_classified_standalone.html")

print(os.path.exists(geojson_path))
print(os.path.exists(html_path))

True
True


**The actual embedding bug.** Inserting embedded data right before `</body>`, but the map's own logic script runs earlier and checks for that data before it exists — the check always failed and fell through to a broken fetch.

In [59]:
with open(geojson_path, "r", encoding="utf-8") as f:
    geojson_data = json.load(f)

with open(html_path, "r", encoding="utf-8") as f:
    html_content = f.read()

embed_script = f"<script>window.EMBEDDED_HIGHWAY_DATA = {json.dumps(geojson_data)};</script>\n"
merged_html = html_content.replace("</body>", embed_script + "</body>")

with open(output_path, "w", encoding="utf-8") as f:
    f.write(merged_html)

print("Done - standalone file created at:", output_path)

Done - standalone file created at: C:\Users\ajots\OneDrive\Documents\ArcGIS\Projects\GTA_Highway_Traffic_Volumes\data processed\highways_classified_standalone.html


**Real fix** — inserting the embedded data before the Leaflet script tag instead, so it's defined before anything tries to read it.

In [60]:
with open(geojson_path, "r", encoding="utf-8") as f:
    geojson_data = json.load(f)

with open(html_path, "r", encoding="utf-8") as f:
    html_content = f.read()

embed_script = f"<script>window.EMBEDDED_HIGHWAY_DATA = {json.dumps(geojson_data)};</script>\n"

# Insert BEFORE the Leaflet script tag, not before </body> - order matters!
insertion_point = '<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.js"></script>'
merged_html = html_content.replace(insertion_point, embed_script + insertion_point)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(merged_html)

print("Done - standalone file created at:", output_path)

Done - standalone file created at: C:\Users\ajots\OneDrive\Documents\ArcGIS\Projects\GTA_Highway_Traffic_Volumes\data processed\highways_classified_standalone.html


Checking every field on `highways_classified_v2` before building the Highway Overview feature — confirming `L_STANDARD_MUNICIPALITY`/`R_STANDARD_MUNICIPALITY` survived the whole pipeline.

In [61]:
fields = arcpy.ListFields("highways_classified_v2")
for f in fields:
    print(f.name, f.type)

OBJECTID OID
Shape Geometry
Join_Count Integer
TARGET_FID Integer
ROAD_NET_ELEMENT_ID Integer
FULL_STREET_NAME String
ABBREVIATED_STREET_NAME String
ALT_STREET_NAME String
DIRECTIONAL_PREFIX String
STREET_TYPE_PREFIX String
STREET_NAME_BODY String
STREET_TYPE_SUFFIX String
DIRECTIONAL_SUFFIX String
ROAD_CLASS String
ROAD_ELEMENT_TYPE String
L_FIRST_HOUSE_NUMBER Integer
L_LAST_HOUSE_NUMBER Integer
L_HOUSE_NUMBER_STRUCTURE String
L_STANDARD_MUNICIPALITY String
R_FIRST_HOUSE_NUMBER Integer
R_LAST_HOUSE_NUMBER Integer
R_HOUSE_NUMBER_STRUCTURE String
R_STANDARD_MUNICIPALITY String
DIRECTION_OF_TRAFFIC_FLOW String
SPEED_LIMIT Integer
NUMBER_OF_LANES Integer
PAVEMENT_STATUS String
JURISDICTION String
ROUTE_NUMBER String
SHIELD_TYPE String
GEOMETRY_UPDATE_DATETIME Date
EFFECTIVE_DATETIME Date
LENGTH Double
segment_type String
ORIG_FID Integer
ORIG_SEQ Integer
MTO_REGION String
MTO_DISTRI String
OPP_DISTRI String
COUNTY Integer
HWY String
LHRS Integer
LENGTH_1 Double
OFFSET Integer
DESCRIPTIO S

**Building the per-highway stats** for the overview panel — cities, average AADT, busiest/quietest points, interchange counts, and (buggy at this point) total length using the stale `LENGTH` field.

In [62]:
import json

target_highways = ["400", "401", "403", "404", "410", "427", "QEW"]
highway_stats = {}

with arcpy.da.SearchCursor("highways_classified_v2",
    ["ROUTE_NUMBER", "segment_type", "L_STANDARD_MUNICIPALITY", "R_STANDARD_MUNICIPALITY",
     "LENGTH", "AADT_2021_v2", "DESCRIPTIO", "FULL_STREET_NAME"]) as cursor:
    rows = list(cursor)

for hwy in target_highways:
    cities = set()
    total_length = 0.0
    aadt_values = []
    busiest = {"aadt": -1, "desc": None}
    quietest = {"aadt": float("inf"), "desc": None}
    ramp_count = 0

    for r in rows:
        route_num, seg_type, l_muni, r_muni, length, aadt, desc, street = r

        # Mainline stats: match if this highway appears in the (possibly combined) ROUTE_NUMBER
        if seg_type == "Mainline" and route_num and hwy in [x.strip() for x in route_num.split(";")]:
            if l_muni: cities.add(l_muni)
            if r_muni: cities.add(r_muni)
            if length: total_length += length
            if aadt is not None:
                aadt_values.append(aadt)
                if aadt > busiest["aadt"]:
                    busiest = {"aadt": aadt, "desc": desc}
                if aadt < quietest["aadt"]:
                    quietest = {"aadt": aadt, "desc": desc}

        # Ramp count: match if this highway's name appears in the ramp's street name
        if seg_type == "Ramp" and street:
            street_upper = street.upper()
            hwy_label = "QUEEN ELIZABETH WAY" if hwy == "QEW" else f"HIGHWAY {hwy}"
            if hwy_label in street_upper:
                ramp_count += 1

    highway_stats[hwy] = {
        "cities": sorted(cities),
        "total_length_km": round(total_length, 1),
        "avg_aadt": round(sum(aadt_values) / len(aadt_values)) if aadt_values else None,
        "busiest": busiest,
        "quietest": quietest,
        "interchange_count": ramp_count
    }

for hwy, stats in highway_stats.items():
    print(hwy, "->", stats)

400 -> {'cities': ['City of Barrie', 'City of Toronto', 'City of Vaughan', 'Town of Bradford West Gwillimbury', 'Town of Innisfil', 'Town of Parry Sound', 'Township of Georgian Bay', 'Township of King', 'Township of McDougall', 'Township of Oro-Medonte', 'Township of Seguin', 'Township of Severn', 'Township of Springwater', 'Township of Tay', 'Wahta Mohawk Territory'], 'total_length_km': 562567.1, 'avg_aadt': 77823, 'busiest': {'aadt': 424600.0, 'desc': 'HWY 400 IC-359-NORTH YORK'}, 'quietest': {'aadt': 8100.0, 'desc': 'GEORGIAN BAY RD - CROOKED BAY RD IC 168'}, 'interchange_count': 356}
401 -> {'cities': ['City of Belleville', 'City of Brockville', 'City of Cambridge', 'City of Cornwall', 'City of Kingston', 'City of London', 'City of Mississauga', 'City of Oshawa', 'City of Pickering', 'City of Quinte West', 'City of Toronto', 'City of Windsor', 'City of Woodstock', 'Municipality of Brighton', 'Municipality of Chatham-Kent', 'Municipality of Clarington', 'Municipality of Dutton/Dunwi

First attempt fixing the length bug with `CalculateGeometryAttributes` — used the wrong parameter name (`GEODESIC_LENGTH` instead of `LENGTH_GEODESIC`) and it errored immediately.

In [63]:
arcpy.management.CalculateGeometryAttributes(
    in_features="highways_classified_v2",
    geometry_property=[["seg_length_km", "GEODESIC_LENGTH"]],
    length_unit="KILOMETERS"
)

<class 'arcgisscripting.ExecuteError'>: Failed to execute. Parameters are not valid.
ERROR 000800: The value is not a member of LENGTH_GEODESIC | LINE_BEARING | PART_COUNT | POINT_COUNT | CURVE_COUNT | LINE_START_X | LINE_START_Y | LINE_START_M | LINE_END_X | LINE_END_Y | LINE_END_M | CENTROID_X | CENTROID_Y | CENTROID_M | INSIDE_X | INSIDE_Y | INSIDE_M | EXTENT_MIN_X | EXTENT_MIN_Y | EXTENT_MAX_X | EXTENT_MAX_Y.
Failed to execute (CalculateGeometryAttributes).


**Corrected parameter name.** Recalculates each stretch's real current length from its geometry, instead of the stale pre-split `LENGTH` attribute.

In [64]:
arcpy.management.CalculateGeometryAttributes(
    in_features="highways_classified_v2",
    geometry_property=[["seg_length_km", "LENGTH_GEODESIC"]],
    length_unit="KILOMETERS"
)

<Result 'highways_classified_v2'>

Re-running the length aggregation with the corrected field. Still roughly double real-world figures at this point.

In [65]:
with arcpy.da.SearchCursor("highways_classified_v2",
    ["ROUTE_NUMBER", "segment_type", "seg_length_km"]) as cursor:
    rows = list(cursor)

for hwy in target_highways:
    total_length = 0.0
    for route_num, seg_type, seg_len in rows:
        if seg_type == "Mainline" and route_num and hwy in [x.strip() for x in route_num.split(";")]:
            if seg_len:
                total_length += seg_len
    highway_stats[hwy]["total_length_km"] = round(total_length, 1)

for hwy in target_highways:
    print(hwy, highway_stats[hwy]["total_length_km"])

400 452.4
401 1784.2
403 253.8
404 100.9
410 43.8
427 58.4
QEW 278.4


Checking `DIRECTION_OF_TRAFFIC_FLOW` — confirmed divided highways are digitized as two separate carriageway lines, explaining the ~2x length inflation.

In [66]:
with arcpy.da.SearchCursor("highways_classified_v2", ["DIRECTION_OF_TRAFFIC_FLOW"]) as cursor:
    from collections import Counter
    tally = Counter(row[0] for row in cursor)

for val, n in tally.items():
    print(repr(val), n)

'Negative' 2759
'Positive' 2740
'Both' 40


Halving the total length to correct for the double-digitized carriageways — landed close to real-world lengths after this.

In [67]:
for hwy in target_highways:
    highway_stats[hwy]["total_length_km"] = round(highway_stats[hwy]["total_length_km"] / 2, 1)

for hwy in target_highways:
    print(hwy, highway_stats[hwy]["total_length_km"])

400 226.2
401 892.1
403 126.9
404 50.5
410 21.9
427 29.2
QEW 139.2


Final embed attempt for the single-year version — geojson + highway_stats baked into one standalone file.

In [68]:
import json
import arcpy
import os

aprx = arcpy.mp.ArcGISProject("CURRENT")
project_folder = aprx.homeFolder

geojson_path = os.path.join(project_folder, "data processed", "highways_classified.geojson")
html_path = os.path.join(project_folder, "data processed", "highways_classified.html")  # the NEW download from above
output_path = os.path.join(project_folder, "data processed", "highways_classified_standalone.html")

with open(geojson_path, "r", encoding="utf-8") as f:
    geojson_data = json.load(f)

with open(html_path, "r", encoding="utf-8") as f:
    html_content = f.read()

geojson_embed = f"<script>window.EMBEDDED_HIGHWAY_DATA = {json.dumps(geojson_data)};</script>\n"
stats_embed = f"<script>window.HIGHWAY_STATS = {json.dumps(highway_stats)};</script>\n"

insertion_point = '<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.js"></script>'
merged_html = html_content.replace(insertion_point, geojson_embed + stats_embed + insertion_point)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(merged_html)

print("Done:", output_path)

Done: C:\Users\ajots\OneDrive\Documents\ArcGIS\Projects\GTA_Highway_Traffic_Volumes\data processed\highways_classified_standalone.html


Reworked version of the embed script, separated into its own cleaner cell with `data_processed` as its own variable.

In [1]:
import json
import arcpy
import os

aprx = arcpy.mp.ArcGISProject("CURRENT")
project_folder = aprx.homeFolder
data_processed = os.path.join(project_folder, "data processed")

html_path = os.path.join(data_processed, "highways_classified.html")
output_path = os.path.join(data_processed, "index.html")

with open(html_path, "r", encoding="utf-8") as f:
    html_content = f.read()

stats_embed = f"<script>window.HIGHWAY_STATS = {json.dumps(highway_stats)};</script>\n"

insertion_point = '<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.js"></script>'
merged_html = html_content.replace(insertion_point, stats_embed + insertion_point)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(merged_html)

print("Created:", output_path)

<class 'NameError'>: name 'highway_stats' is not defined

**Kernel reset again.** Reconnecting to the workspace and rebuilding `highway_stats` from scratch since the variable got wiped.

In [2]:
import arcpy
import os
import json
from collections import Counter

# Reconnect to the workspace
aprx = arcpy.mp.ArcGISProject("CURRENT")
arcpy.env.workspace = aprx.defaultGeodatabase
arcpy.env.overwriteOutput = True
project_folder = aprx.homeFolder
data_processed = os.path.join(project_folder, "data processed")

TARGET_ROUTE_NUMBERS = ["400", "401", "403", "404", "410", "427", "QEW"]
STREET_NAMES = {
    "400": "HIGHWAY 400", "401": "HIGHWAY 401", "403": "HIGHWAY 403",
    "404": "HIGHWAY 404", "410": "HIGHWAY 410", "427": "HIGHWAY 427",
    "QEW": "QUEEN ELIZABETH WAY"
}

# Rebuild highway_stats from your final layer (already saved in the geodatabase)
highway_stats = {}

with arcpy.da.SearchCursor("highways_classified_v2",
    ["ROUTE_NUMBER", "segment_type", "L_STANDARD_MUNICIPALITY", "R_STANDARD_MUNICIPALITY",
     "seg_length_km", "AADT_2021_v2", "DESCRIPTIO", "FULL_STREET_NAME"]) as cursor:
    rows = list(cursor)

for hwy in TARGET_ROUTE_NUMBERS:
    cities = set()
    total_length = 0.0
    aadt_values = []
    busiest = {"aadt": -1, "desc": None}
    quietest = {"aadt": float("inf"), "desc": None}
    ramp_count = 0

    for route_num, seg_type, l_muni, r_muni, seg_len, aadt, desc, street in rows:
        if seg_type == "Mainline" and route_num and hwy in [x.strip() for x in route_num.split(";")]:
            if l_muni: cities.add(l_muni)
            if r_muni: cities.add(r_muni)
            if seg_len: total_length += seg_len
            if aadt is not None:
                aadt_values.append(aadt)
                if aadt > busiest["aadt"]:
                    busiest = {"aadt": aadt, "desc": desc}
                if aadt < quietest["aadt"]:
                    quietest = {"aadt": aadt, "desc": desc}

        if seg_type == "Ramp" and street and STREET_NAMES[hwy] in street.upper():
            ramp_count += 1

    highway_stats[hwy] = {
        "cities": sorted(cities),
        "total_length_km": round(total_length / 2, 1),
        "avg_aadt": round(sum(aadt_values) / len(aadt_values)) if aadt_values else None,
        "busiest": busiest,
        "quietest": quietest,
        "interchange_count": ramp_count
    }

print("Rebuilt highway_stats for:", list(highway_stats.keys()))

Rebuilt highway_stats for: ['400', '401', '403', '404', '410', '427', 'QEW']


Checking the exact field names for AADT/length on the layer, since I wasn't sure if it was `AADT_2021` or `AADT_2021_v2` after all the rebuilds.

In [3]:
fields = [f.name for f in arcpy.ListFields("highways_classified_v2")]
print([f for f in fields if "AADT" in f.upper() or "LENGTH" in f.upper()])

['LENGTH', 'LENGTH_1', 'AADT_2021', 'AADT_2021_v2', 'Shape_Length', 'seg_length_km']


Embedding the rebuilt `highway_stats` into a fresh `index.html`.

In [4]:
html_path = os.path.join(data_processed, "highways_classified.html")
output_path = os.path.join(data_processed, "index.html")

with open(html_path, "r", encoding="utf-8") as f:
    html_content = f.read()

stats_embed = f"<script>window.HIGHWAY_STATS = {json.dumps(highway_stats)};</script>\n"

insertion_point = '<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.js"></script>'
merged_html = html_content.replace(insertion_point, stats_embed + insertion_point)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(merged_html)

print("Created:", output_path)

Created: C:\Users\ajots\OneDrive\Documents\ArcGIS\Projects\GTA_Highway_Traffic_Volumes\data processed\index.html


Empty cell.

**Starting the historical time-slider build.** Reloading everything and this time pulling *every* year of AADT data (1988-2021), not just 2021, since I want the slider to show traffic evolving over time.

In [1]:
import arcpy, os, json
import pandas as pd
import numpy as np

aprx = arcpy.mp.ArcGISProject("CURRENT")
arcpy.env.workspace = aprx.defaultGeodatabase
arcpy.env.overwriteOutput = True
project_folder = aprx.homeFolder
data_raw = os.path.join(project_folder, "data raw")
data_processed = os.path.join(project_folder, "data processed")

HWY_CODE_MAP = {"400": 400, "401": 401, "403": 403, "404": 404, "410": 410, "427": 427, "QEW": 1}

# Load ALL years this time, not just 2021
csv_path = os.path.join(data_raw, "Traffic_Volumes_1988-2021.csv")
aadt_df = pd.read_csv(csv_path)

aadt_filtered = aadt_df[
    (aadt_df["Hwy No"].isin(HWY_CODE_MAP.values())) &
    (aadt_df["Year"] != 9999)   # exclude the bogus placeholder year
].copy()

aadt_filtered["AADT_clean"] = aadt_filtered["AADT"].astype(str).str.replace(",", "").astype(float)

print("Years available:", sorted(aadt_filtered["Year"].unique()))
print("Total rows across all years:", len(aadt_filtered))

Years available: [1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2021]
Total rows across all years: 10574


Building a station-level lookup: `"LHRS_OFFSET" -> {year: AADT}` — the core data structure the slider will read from client-side in the browser.

In [2]:
# Build a lookup: "LHRS_OFFSET" -> {year: aadt}
yearly_lookup = {}

for _, row in aadt_filtered.iterrows():
    key = f"{row['LHRS']}_{row['Offset']}"
    year = str(int(row['Year']))
    aadt = row['AADT_clean']
    if key not in yearly_lookup:
        yearly_lookup[key] = {}
    yearly_lookup[key][year] = aadt

print("Distinct stations with historical data:", len(yearly_lookup))
# Peek at one station's full history
sample_key = list(yearly_lookup.keys())[0]
print(sample_key, "->", yearly_lookup[sample_key])

Distinct stations with historical data: 338
10000_0.0 -> {'1988': 14000.0, '1989': 14500.0, '1990': 15100.0, '1991': 15300.0, '1992': 22200.0, '1993': 21500.0, '1994': 20800.0, '1995': 21100.0, '1996': 20700.0, '1997': 21100.0, '1998': 20900.0, '1999': 21900.0, '2000': 22500.0, '2001': 22000.0, '2002': 22100.0, '2003': 19900.0, '2004': 19000.0, '2005': 19000.0, '2006': 18800.0, '2007': 18200.0, '2008': 17400.0, '2009': 16200.0, '2010': 16400.0, '2011': 16600.0, '2012': 16500.0, '2013': 16300.0, '2014': 15300.0, '2015': 14800.0, '2016': 14600.0, '2017': 14400.0, '2018': 14300.0, '2019': 14000.0, '2021': 14800.0}


Getting the *full* set of station locations for my 7 highways, regardless of which specific years they have data for — needed since the slider covers all years, not just 2021.

In [3]:
base_points_path = os.path.join(data_raw, "LHRS_Base_Points_July2015.shp")
arcpy.management.CopyFeatures(base_points_path, "lhrs_base_points_all")

# Filter to our 7 highways only - keep ALL of them regardless of which years they have data
where_clause = "HWY IN ('1','400','401','403','404','410','427')"
arcpy.management.SelectLayerByAttribute("lhrs_base_points_all", "NEW_SELECTION", where_clause)
arcpy.management.CopyFeatures("lhrs_base_points_all", "aadt_stations_all_years")

# Add a station_key field matching the lookup dict's key format
arcpy.management.AddField("aadt_stations_all_years", "station_key", "TEXT", field_length=30)
with arcpy.da.UpdateCursor("aadt_stations_all_years", ["LHRS", "OFFSET", "station_key"]) as cursor:
    for row in cursor:
        row[2] = f"{row[0]}_{row[1]}"
        cursor.updateRow(row)

print("Total station locations for these highways:", arcpy.management.GetCount("aadt_stations_all_years")[0])

Total station locations for these highways: 384


Splitting the highway mainline at this fuller station set (384 locations now, vs. 327 before) and joining each stretch to its nearest station.

In [4]:
arcpy.management.SplitLineAtPoint(
    in_features="highways_mainline_only",
    point_features="aadt_stations_all_years",
    out_feature_class="highways_split_historical",
    search_radius="100 Meters"
)
print("Split stretches:", arcpy.management.GetCount("highways_split_historical")[0])

# Join each stretch to its nearest station, carrying over the station_key field
arcpy.analysis.SpatialJoin(
    target_features="highways_split_historical",
    join_features="aadt_stations_all_years",
    out_feature_class="highways_historical_joined",
    join_operation="JOIN_ONE_TO_ONE",
    match_option="CLOSEST",
    search_radius="50000 Meters"
)

with arcpy.da.SearchCursor("highways_historical_joined", ["station_key"]) as cursor:
    null_count = sum(1 for row in cursor if row[0] is None)
print("Unmatched stretches:", null_count)

Split stretches: 3033
Unmatched stretches: 0


Merging ramps back in for the historical version too — ramps get no `station_key`, so they'll just render fixed gray regardless of which year is selected.

In [5]:
arcpy.management.SelectLayerByAttribute("highways_filtered_v2", "NEW_SELECTION", "segment_type = 'Ramp'")
arcpy.management.CopyFeatures("highways_filtered_v2", "highways_ramps_historical")
arcpy.management.AddField("highways_ramps_historical", "station_key", "TEXT", field_length=30)
# Ramps get no station_key (stays null) - handled as fixed gray in the map regardless of year

arcpy.management.Merge(["highways_historical_joined", "highways_ramps_historical"], "highways_historical_final")
print("Final layer:", arcpy.management.GetCount("highways_historical_final")[0])

Final layer: 7818


Reprojecting to WGS84 and exporting the historical GeoJSON — this one carries a `station_key` per stretch instead of a baked-in AADT/tier, since the tier now gets computed live in the browser based on whatever year the slider is on.

In [6]:
arcpy.management.Project(
    in_dataset="highways_historical_final",
    out_dataset="highways_historical_wgs84",
    out_coor_system=arcpy.SpatialReference(4326)
)

historical_geojson_path = os.path.join(data_processed, "highways_historical.geojson")
arcpy.conversion.FeaturesToJSON(
    in_features="highways_historical_wgs84",
    out_json_file=historical_geojson_path,
    geoJSON="GEOJSON"
)
print("Exported:", historical_geojson_path)

Exported: C:\Users\ajots\OneDrive\Documents\ArcGIS\Projects\GTA_Highway_Traffic_Volumes\data processed\highways_historical.geojson


Saving the year-by-year lookup dictionary as its own small JSON file, separate from the geometry.

In [7]:
lookup_path = os.path.join(data_processed, "yearly_lookup.json")
with open(lookup_path, "w", encoding="utf-8") as f:
    json.dump(yearly_lookup, f)

print("Lookup file size (KB):", os.path.getsize(lookup_path) / 1024)

Lookup file size (KB): 183.5283203125


First attempt building `index.html` for the historical/slider version - embedding just the highway_stats, keeping the (much larger) historical geojson and lookup as separate fetched files instead of embedding everything, learned from the file-size pain of the single-year version.

In [8]:
import json, os

html_path = os.path.join(data_processed, "highways_classified.html")   # the NEW slider template
output_path = os.path.join(data_processed, "index.html")               # overwrite the old one

with open(html_path, "r", encoding="utf-8") as f:
    html_content = f.read()

stats_embed = f"<script>window.HIGHWAY_STATS = {json.dumps(highway_stats)};</script>\n"

insertion_point = '<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.js"></script>'
merged_html = html_content.replace(insertion_point, stats_embed + insertion_point)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(merged_html)

print("New index.html created (with slider + stats):", output_path)

<class 'NameError'>: name 'highway_stats' is not defined

Kernel reset yet again — reconnecting and rebuilding `highway_stats` one more time for the slider version.

In [9]:
import arcpy, os, json
from collections import Counter

aprx = arcpy.mp.ArcGISProject("CURRENT")
arcpy.env.workspace = aprx.defaultGeodatabase
arcpy.env.overwriteOutput = True
project_folder = aprx.homeFolder
data_processed = os.path.join(project_folder, "data processed")

TARGET_ROUTE_NUMBERS = ["400", "401", "403", "404", "410", "427", "QEW"]
STREET_NAMES = {
    "400": "HIGHWAY 400", "401": "HIGHWAY 401", "403": "HIGHWAY 403",
    "404": "HIGHWAY 404", "410": "HIGHWAY 410", "427": "HIGHWAY 427",
    "QEW": "QUEEN ELIZABETH WAY"
}

highway_stats = {}

with arcpy.da.SearchCursor("highways_classified_v2",
    ["ROUTE_NUMBER", "segment_type", "L_STANDARD_MUNICIPALITY", "R_STANDARD_MUNICIPALITY",
     "seg_length_km", "AADT_2021_v2", "DESCRIPTIO", "FULL_STREET_NAME"]) as cursor:
    rows = list(cursor)

for hwy in TARGET_ROUTE_NUMBERS:
    cities = set()
    total_length = 0.0
    aadt_values = []
    busiest = {"aadt": -1, "desc": None}
    quietest = {"aadt": float("inf"), "desc": None}
    ramp_count = 0

    for route_num, seg_type, l_muni, r_muni, seg_len, aadt, desc, street in rows:
        if seg_type == "Mainline" and route_num and hwy in [x.strip() for x in route_num.split(";")]:
            if l_muni: cities.add(l_muni)
            if r_muni: cities.add(r_muni)
            if seg_len: total_length += seg_len
            if aadt is not None:
                aadt_values.append(aadt)
                if aadt > busiest["aadt"]:
                    busiest = {"aadt": aadt, "desc": desc}
                if aadt < quietest["aadt"]:
                    quietest = {"aadt": aadt, "desc": desc}

        if seg_type == "Ramp" and street and STREET_NAMES[hwy] in street.upper():
            ramp_count += 1

    highway_stats[hwy] = {
        "cities": sorted(cities),
        "total_length_km": round(total_length / 2, 1),
        "avg_aadt": round(sum(aadt_values) / len(aadt_values)) if aadt_values else None,
        "busiest": busiest,
        "quietest": quietest,
        "interchange_count": ramp_count
    }

print("Rebuilt for:", list(highway_stats.keys()))

Rebuilt for: ['400', '401', '403', '404', '410', '427', 'QEW']


Embedding the rebuilt stats into `index.html` again.

In [10]:
html_path = os.path.join(data_processed, "highways_classified.html")
output_path = os.path.join(data_processed, "index.html")

with open(html_path, "r", encoding="utf-8") as f:
    html_content = f.read()

stats_embed = f"<script>window.HIGHWAY_STATS = {json.dumps(highway_stats)};</script>\n"

insertion_point = '<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.js"></script>'
merged_html = html_content.replace(insertion_point, stats_embed + insertion_point)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(merged_html)

print("Created:", output_path)

Created: C:\Users\ajots\OneDrive\Documents\ArcGIS\Projects\GTA_Highway_Traffic_Volumes\data processed\index.html


Same embed, but added an `assert` this time to fail loudly if the wrong HTML template (without the slider markup) accidentally gets used again — bit me once already.

In [11]:
html_path = os.path.join(data_processed, "highways_classified.html")
output_path = os.path.join(data_processed, "index.html")

with open(html_path, "r", encoding="utf-8") as f:
    html_content = f.read()

# Sanity check before embedding - confirm we're using the right template
assert "yearSlider" in html_content, "Wrong template - slider markup not found in source file"

stats_embed = f"<script>window.HIGHWAY_STATS = {json.dumps(highway_stats)};</script>\n"
insertion_point = '<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.js"></script>'
merged_html = html_content.replace(insertion_point, stats_embed + insertion_point)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(merged_html)

print("Created:", output_path)

Created: C:\Users\ajots\OneDrive\Documents\ArcGIS\Projects\GTA_Highway_Traffic_Volumes\data processed\index.html


**Found a real bug via the browser console**: `json.dump()` happily writes Python's `NaN` for invalid values, but that's not actually valid JSON — browsers reject it outright. Rebuilding the yearly lookup while explicitly skipping any NaN AADT values instead of writing them into the file.

In [12]:
import math

# Rebuild the lookup, skipping any NaN AADT values entirely
yearly_lookup = {}

for _, row in aadt_filtered.iterrows():
    aadt = row['AADT_clean']
    if aadt is None or (isinstance(aadt, float) and math.isnan(aadt)):
        continue  # skip invalid entries instead of writing NaN into the JSON

    key = f"{row['LHRS']}_{row['Offset']}"
    year = str(int(row['Year']))
    if key not in yearly_lookup:
        yearly_lookup[key] = {}
    yearly_lookup[key][year] = aadt

print("Distinct stations:", len(yearly_lookup))

# Save again, overwriting the broken file
lookup_path = os.path.join(data_processed, "yearly_lookup.json")
with open(lookup_path, "w", encoding="utf-8") as f:
    json.dump(yearly_lookup, f)

print("Fixed lookup saved:", os.path.getsize(lookup_path) / 1024, "KB")

Distinct stations: 332
Fixed lookup saved: 183.0341796875 KB


**Found a second, sneakier bug.** Even after removing NaNs, values were still coming back `undefined` in the browser. Turned out `Offset` is a float in the AADT CSV (`0.0`) but an integer in the shapefile (`0`) — so the lookup keys never actually matched even though the data was correct. Casting both to `int()` before building the key fixed it for real.

In [13]:
import math

yearly_lookup = {}

for _, row in aadt_filtered.iterrows():
    aadt = row['AADT_clean']
    if aadt is None or (isinstance(aadt, float) and math.isnan(aadt)):
        continue

    # Cast Offset to int to match the shapefile's Integer field format exactly
    key = f"{int(row['LHRS'])}_{int(row['Offset'])}"
    year = str(int(row['Year']))
    if key not in yearly_lookup:
        yearly_lookup[key] = {}
    yearly_lookup[key][year] = aadt

print("Distinct stations:", len(yearly_lookup))
sample_key = list(yearly_lookup.keys())[0]
print(sample_key, "->", yearly_lookup[sample_key])

lookup_path = os.path.join(data_processed, "yearly_lookup.json")
with open(lookup_path, "w", encoding="utf-8") as f:
    json.dump(yearly_lookup, f)
print("Saved:", os.path.getsize(lookup_path) / 1024, "KB")


Distinct stations: 332
10000_0 -> {'1988': 14000.0, '1989': 14500.0, '1990': 15100.0, '1991': 15300.0, '1992': 22200.0, '1993': 21500.0, '1994': 20800.0, '1995': 21100.0, '1996': 20700.0, '1997': 21100.0, '1998': 20900.0, '1999': 21900.0, '2000': 22500.0, '2001': 22000.0, '2002': 22100.0, '2003': 19900.0, '2004': 19000.0, '2005': 19000.0, '2006': 18800.0, '2007': 18200.0, '2008': 17400.0, '2009': 16200.0, '2010': 16400.0, '2011': 16600.0, '2012': 16500.0, '2013': 16300.0, '2014': 15300.0, '2015': 14800.0, '2016': 14600.0, '2017': 14400.0, '2018': 14300.0, '2019': 14000.0, '2021': 14800.0}
Saved: 182.3857421875 KB


Empty trailing cell.